# EDGAR Credit Clustering v3 — Clean src-driven notebook

This notebook is intentionally thin. It orchestrates the workflow and delegates reusable logic to `src.credit_clustering`:

- EDGAR concept/SIC mapping: `edgar_concepts.py`
- Feature engineering: `features.py`
- Clustering: `clustering.py`
- Cluster profiling: `profiling.py`
- Artifact building/saving: `artifacts.py`

The notebook should not define reusable methods.

In [1]:
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)

In [20]:
BASE_URL = "https://pub-a6c3a3e1a0f546beb4be7cc34fd647d1.r2.dev/raw_financial_facts_parquet"

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DUCKDB_DIR = PROJECT_ROOT / "data" / "duckdb"
DUCKDB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DUCKDB_DIR / "financials.duckdb"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)

CURRENT_OUTPUT_DIR = OUTPUT_DIR / "credit_clustering_outputs_v3"
CURRENT_OUTPUT_DIR.mkdir(exist_ok=True)

ARTIFACT_PATH = MODEL_DIR / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"

In [3]:
#Import your modules here

from src.credit_clustering import (
    # Config
    PUBLIC_COMPANY_MIN_ASSETS,
    SCORECARD_CLUSTER_FEATURES,
    SCORECARD_COMPONENT_FEATURES,
    RATIO_COLS,
    INTERPRET_FEATURES,
    DEFAULT_SEGMENT_COL,
    DEFAULT_TARGET_SEGMENTS,
    DEFAULT_N_CLUSTERS,
    DEFAULT_MIN_ROWS_PER_SEGMENT,
    cluster_segments,

    # EDGAR concepts and panel construction
    EDGAR_CONCEPT_MAP,
    create_issuer_year_facts_table,
    build_issuer_year_panel,

    # Feature engineering
    engineer_private_company_features,

    # Clustering
    cluster_segments,
    evaluate_segments_k_range,

    # Profiling
    build_cluster_profile,
    build_cluster_medians,
    build_feature_extremes,
    build_industry_cluster_mix,
    add_rating_style_labels,
    build_rating_label_maps,
    representatives,
    merge_profile_with_representatives,

    # Artifacts
    build_credit_clustering_artifact,
    save_artifact,
    summarize_artifact,
)

## 1. Connect to EDGAR raw facts

The raw EDGAR-derived facts are read from remote Parquet files into a DuckDB view.

In [4]:
parquet_urls = [
    f"{BASE_URL}/part_{i:05d}.parquet"
    for i in range(401)
]

con = duckdb.connect(DB_PATH)

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(f"""
CREATE OR REPLACE VIEW raw_facts AS
SELECT *
FROM read_parquet({parquet_urls})
""")

schema = con.execute("DESCRIBE raw_facts").df()
schema.head()

,column_name,column_type,null,key,default,extra
0,concept,VARCHAR,YES,None,None,None
1,label,VARCHAR,YES,None,None,None
2,value,DOUBLE,YES,None,None,None
3,numeric_value,DOUBLE,YES,None,None,None
4,unit,VARCHAR,YES,None,None,None


In [5]:
raw_summary = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year
FROM raw_facts
""").df()

raw_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,tickers,concepts,min_year,max_year
0,100149899,8194,13256,0,44012


## 2. Build issuer-year facts and panel

The EDGAR concept map lives in `src.credit_clustering.edgar_concepts`. The notebook only calls the extraction helpers.

In [6]:
facts_summary, value_col, sort_col = create_issuer_year_facts_table(
    con=con,
    schema=schema,
    concept_map=EDGAR_CONCEPT_MAP,
    start_year=2020,
    end_year=2025,
    fiscal_period="FY",
)

print("value_col:", value_col, "| sort_col:", sort_col)
display(facts_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

value_col: numeric_value | sort_col: period_end


,canonical_feature,row_count,ticker_count
0,net_income,36055,7171
1,assets,36044,7147
2,cfo,35527,7106
3,equity,35594,7048
4,cash,34942,7035
5,depreciation_amortization,31723,6491
6,liabilities,31638,6462
7,revenue,27296,5841
8,operating_income,27902,5778
9,ppe,26720,5771


In [8]:
panel = build_issuer_year_panel(
    con=con,
    concept_map=EDGAR_CONCEPT_MAP,
)

print("Panel shape:", panel.shape)
display(panel.head())

Panel shape: (36290, 29)


,ticker,cik,company_name,sic,fiscal_year,assets,assets_current,capex,cash,cfo,depreciation_amortization,equity,goodwill,gross_profit,interest_expense,inventory,liabilities,liabilities_current,long_term_debt,net_income,operating_income,ppe,rd,receivables,revenue,sga,short_term_debt,major_sector,financial_flag
0,SNDR,1692063,"Schneider National, Inc.",4213,2023,4.437700e+09,1.230600e+09,535100000.0,315250000.0,680000000.0,296200000.0,2.630500e+09,240500000.0,NaN,12500000.0,85450000.0,1.540700e+09,621550000.0,162500000.0,405400000.0,533700000.0,2.430850e+09,NaN,609700000.0,5.608700e+09,NaN,85000000.0,Transportation / Utilities,Non-financial
1,ATVK,1530185,"Ameritek Ventures, Inc.",3714,2024,5.490315e+06,6.975850e+04,0.0,751.0,150516.0,57388.0,2.983690e+05,1978195.5,NaN,173767.5,NaN,3.058657e+06,1580727.5,24596.0,2218944.5,172937.0,0.000000e+00,405454.0,66190.0,8.138690e+05,NaN,21000.0,Manufacturing / Industrials,Non-financial
2,MEC,1766368,"Mayville Engineering Company, Inc.",3460,2025,5.046050e+08,1.206790e+08,12098000.0,854000.0,40363000.0,18527000.0,2.353575e+08,92650000.0,NaN,NaN,57077000.0,2.583600e+08,69680500.0,NaN,7844000.0,20191000.0,1.532620e+08,NaN,NaN,5.816040e+08,NaN,NaN,Manufacturing / Industrials,Non-financial
3,WLDSW,1887673,Wearable Devices Ltd.,3576,2022,6.334500e+06,6.189000e+06,36000.0,1427000.0,-2103000.0,11000.0,-4.450000e+05,NaN,NaN,NaN,NaN,9.410000e+05,894000.0,NaN,-2614000.0,-2669000.0,5.550000e+04,1411000.0,NaN,5.700000e+04,NaN,NaN,Manufacturing / Industrials,Non-financial
4,CDXS,1200375,"CODEXIS, INC.",2860,2021,2.340145e+08,1.731555e+08,3748000.0,116797000.0,-14267000.0,1950000.0,1.349540e+08,3241000.0,NaN,NaN,1062000.0,6.676750e+07,29176000.0,NaN,-21279000.0,-22697000.0,1.551000e+07,44185000.0,19423500.0,6.905600e+07,35049000.0,NaN,Manufacturing / Industrials,Non-financial


In [9]:
panel.groupby(["major_sector", "fiscal_year"]).size().unstack(fill_value=0)

fiscal_year,2020,2021,2022,2023,2024,2025
major_sector,,,,,,
Agriculture,23,33,38,42,42,34
Construction,53,62,66,70,76,80
Finance / Insurance / Real Estate,1441,1522,1582,1621,1654,1663
Manufacturing / Industrials,1830,2099,2237,2297,2365,2253
Mining / Energy,213,225,248,254,246,239
Services,875,1060,1120,1156,1228,1187
Transportation / Utilities,425,450,470,470,487,479
Wholesale / Retail,326,378,388,400,410,373


## 3. Engineer credit scorecard features

Feature construction is centralized in `features.py` so training and private-company scoring use the same logic.

In [10]:
model_df = engineer_private_company_features(
    panel,
    winsor_caps=None,
    fx_to_model_currency=1.0,
)

# Training/reference universe filter.
# The threshold is controlled in config.py.
model_df = model_df.loc[
    model_df["assets"].ge(PUBLIC_COMPANY_MIN_ASSETS)
].copy()

print("Model rows:", len(model_df))
display(model_df[SCORECARD_CLUSTER_FEATURES].describe().T)

Model rows: 34299


,count,mean,std,min,25%,50%,75%,max
structural_distress_risk,34299.0,0.128663,0.334831,0.0,0.000000,0.000000,0.000000,1.0
earnings_risk,34117.0,0.526047,0.388227,0.0,0.175940,0.458806,1.000000,1.0
operating_cashflow_risk,33670.0,0.516182,0.403568,0.0,0.038914,0.568563,1.000000,1.0
liquidity_risk,25178.0,0.341250,0.353563,0.0,0.000000,0.208893,0.658163,1.0
leverage_risk,16415.0,0.323732,0.283252,0.0,0.058439,0.276197,0.527413,1.0
debt_service_risk,7791.0,0.504068,0.397345,0.0,0.063221,0.528076,0.955778,1.0


In [11]:
feature_availability = (
    model_df[SCORECARD_CLUSTER_FEATURES + SCORECARD_COMPONENT_FEATURES]
    .notna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("availability_pct")
)

feature_availability

,availability_pct
structural_distress_risk,1.000000
negative_equity_flag,1.000000
liabilities_exceed_assets_flag,1.000000
earnings_risk,0.994694
profitability_risk,0.994694
equity_buffer_risk,0.983323
operating_cashflow_risk,0.981661
cashflow_risk,0.981661
cash_buffer_risk,0.970407
liabilities_risk,0.878101


## 4. Cluster non-financial issuers

The clustering code lives in `clustering.py`. The model uses bounded directional risk factors, not raw accounting values.

In [13]:
clustered_panel, metrics_df, segment_artifacts = cluster_segments(
    model_df,
    segment_col=DEFAULT_SEGMENT_COL,
    n_clusters=DEFAULT_N_CLUSTERS,
    min_rows=DEFAULT_MIN_ROWS_PER_SEGMENT,
    cluster_only_segments=DEFAULT_TARGET_SEGMENTS,
)

print("Clustered rows:", len(clustered_panel))
display(metrics_df)

Clustered rows: 24003


,segment,status,rows,features,feature_list,silhouette,calinski_harabasz,davies_bouldin,feature_availability,min_non_null_features
0,Financial,skipped_not_target_segment,9271,0,[],NaN,NaN,NaN,NaN,NaN
1,Non-financial,clustered,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",0.360757,13416.412028,1.211214,"{'structural_distress_risk': 1.0, 'earnings_risk': 0.9956448777369347, 'operating_cashflow_risk': 0.9817004954451015, 'liquidity_risk': 0.9513744606041233, ...",4.0


In [15]:
cluster_sizes = (
    clustered_panel
    .groupby([DEFAULT_SEGMENT_COL, "cluster"])
    .size()
    .reset_index(name="issuer_years")
    .sort_values([DEFAULT_SEGMENT_COL, "cluster"])
)

cluster_sizes

,financial_flag,cluster,issuer_years
0,Non-financial,0,6593
1,Non-financial,1,4341
2,Non-financial,2,7087
3,Non-financial,3,3350
4,Non-financial,4,2632


## 5. Build cluster profiles and rating-style labels

The interpretation layer lives in `profiling.py`.

In [16]:
cluster_profile = build_cluster_profile(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_medians = build_cluster_medians(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

feature_extremes = build_feature_extremes(
    clustered_panel,
)

industry_cluster_mix = build_industry_cluster_mix(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_profile_ranked = add_rating_style_labels(
    cluster_profile,
    segment_col=DEFAULT_SEGMENT_COL,
)

rating_labels_by_cluster, risk_rank_by_cluster = build_rating_label_maps(
    cluster_profile_ranked,
    segment_name="Non-financial",
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_representatives = representatives(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_profile_ranked_with_reps = merge_profile_with_representatives(
    cluster_profile_ranked,
    cluster_representatives,
    segment_col=DEFAULT_SEGMENT_COL,
)

display(cluster_profile_ranked_with_reps)

,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label,representative_tickers,sample_companies,sample_years
0,Non-financial,2,7087,1949,21.193320,0.453773,0.235761,0.482906,0.485135,0.117420,0.049063,0.093812,0.610141,2.365987,1.275307,7.003767,0.280442,0.107985,0.376434,0.148135,2.376074,1.251290,10.197074,0.071860,0.067104,0.009368,0.00000,0.087585,0.0,11.200344,1,1 - Low risk / investment-grade-like,"CRTO, TREX, TECH, PKG, UG, NEOG, DXCM, KTEL, INOD, MZTI",Criteo S.A. | TREX CO INC | BIO-TECHNE Corp | PACKAGING CORP OF AMERICA | UNITED GUARDIAN INC,"2021, 2022, 2025, 2023, 2024"
1,Non-financial,1,4341,1241,22.805765,0.670778,0.294820,0.858678,0.301347,0.030583,0.033466,0.081283,0.491363,1.010112,0.350580,3.272810,0.202893,0.128044,0.329196,0.185711,3.372739,2.863957,5.415796,0.266252,0.737445,0.165341,0.00000,0.210794,0.0,32.468255,2,2 - Moderate risk / lower-investment-grade-like,"NOA, YOU, LHX, SGU, SDSYA, MTN, CSTM, GLP-PB, GLP, OXY","North American Construction Group Ltd. | Clear Secure, Inc. | L3HARRIS TECHNOLOGIES, INC. /DE/ | STAR GROUP, L.P. | SOUTH DAKOTA SOYBEAN PROCESSORS LLC","2024, 2022, 2025, 2020"
2,Non-financial,0,6593,2032,18.683641,0.299129,0.126675,0.259263,0.600237,0.277374,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.678366,-1.230223,-0.818926,0.406889,-0.697438,9.780351,3.997271,-23.494511,0.039044,0.000000,1.000000,1.00000,1.000000,0.0,58.333333,3,3 - Elevated risk / leveraged,"PHGE, CLIR, RNLXY, DIBS, CSTL, BHVN, LONA, NERV, YJ, DBVT","BiomX Inc. | ClearSign Technologies Corp | Renalytix plc | 1stdibs.com, Inc. | CASTLE BIOSCIENCES INC","2022, 2025, 2023, 2024, 2020"
3,Non-financial,4,2632,1269,19.570780,0.651929,0.244138,0.837096,0.296061,0.048854,-0.068130,-0.008446,0.458308,1.065221,0.401347,-2.720396,-0.091134,-0.116823,0.295145,-0.061367,9.125257,7.786122,-1.439512,0.289433,0.666973,1.000000,0.80405,0.904431,0.0,62.536025,4,4 - High risk / speculative,"TCX, VNCE, SMWB, CCC, SCOR, GITS, FARM, NISN, AIOS, NOTE","TUCOWS INC /PA/ | VINCE HOLDING CORP. | SIMILARWEB LTD. | CCC Intelligent Solutions Holdings Inc. | COMSCORE, INC.","2024, 2020, 2021, 2023, 2022"
4,Non-financial,3,3350,1572,18.060449,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.459393,9.587202,6.870450,-6.715322,0.693849,0.709803,1.000000,1.00000,1.000000,1.0,77.597603,5,5 - Distressed / near-default proxy,"OCLN, NUAIW, NUAI, LVO, UXIN, CRTD, FOXOW, FOXO, MDAI, MDAIW","ORIGINCLEAR, INC. | New ERA Energy & Digital, Inc. | LiveOne, Inc. | Uxin Ltd | Creatd, Inc.","2020, 2025, 2023, 2024"


In [17]:
display(cluster_medians)
display(feature_extremes)
display(industry_cluster_mix.head(50))

log_assets  liabilities_to_assets  debt_to_assets  \
financial_flag cluster                                                      
Non-financial  0            18.684                  0.299           0.127   
               1            22.806                  0.671           0.295   
               2            21.193                  0.454           0.236   
               3            18.060                  1.086           0.396   
               4            19.571                  0.652           0.244   

                        debt_to_equity  equity_to_assets  cash_to_assets  \
financial_flag cluster                                                     
Non-financial  0                 0.259             0.600           0.277   
               1                 0.859             0.301           0.031   
               2                 0.483             0.485           0.117   
               3                -0.805            -0.209           0.104   
               4                 0.837             0.296           0.049   

                        net_income_to_assets  cfo_to_assets  \
financial_flag cluster                                        
Non-financial  0                      -0.258         -0.198   
               1                       0.033          0.081   
               2                       0.049          0.094   
               3                      -0.277         -0.152   
               4                      -0.068         -0.008   

                        revenue_to_assets  current_ratio  quick_ratio  \
financial_flag cluster                                                  
Non-financial  0                    0.240          4.310        2.130   
               1                    0.491          1.010        0.351   
               2                    0.610          2.366        1.275   
               3                    0.452          0.846        0.394   
               4                    0.458          1.065        0.401   

                        interest_coverage  fcf_to_debt  operating_margin  \
financial_flag cluster                                                     
Non-financial  0                  -27.678       -1.230            -0.819   
               1                    3.273        0.203             0.128   
               2                    7.004        0.280             0.108   
               3                   -7.906       -0.521            -0.529   
               4                   -2.720       -0.091            -0.117   

                        gross_margin  ebitda_margin  debt_to_ebitda  \
financial_flag cluster                                                
Non-financial  0               0.407         -0.697           9.780   
               1               0.329          0.186           3.373   
               2               0.376          0.148           2.376   
               3               0.331         -0.459           9.587   
               4               0.295         -0.061           9.125   

                        net_debt_to_ebitda  ebitda_interest_coverage  \
financial_flag cluster                                                 
Non-financial  0                     3.997                   -23.495   
               1                     2.864                     5.416   
               2                     1.251                    10.197   
               3                     6.870                    -6.715   
               4                     7.786                    -1.440   

                        leverage_risk  liquidity_risk  earnings_risk  \
financial_flag cluster                                                 
Non-financial  0                0.039           0.000          1.000   
               1                0.266           0.737          0.165   
               2                0.072           0.067          0.009   
               3                0.694           0.710          1.000   
               4             

,0.001,0.010,0.050,0.500,0.950,0.990,0.999
log_assets,13.861,14.320,15.561,20.124,24.670,26.001,28.156
liabilities_to_assets,0.004,0.035,0.076,0.510,1.413,4.431,17.012
debt_to_assets,0.000,0.000,0.005,0.246,0.759,1.401,4.898
debt_to_equity,-176.958,-18.145,-2.082,0.459,4.195,19.983,192.857
equity_to_assets,-15.753,-3.319,-0.472,0.383,0.885,1.107,1.474
cash_to_assets,0.000,0.000,0.004,0.109,0.700,0.972,1.350
net_income_to_assets,-10.512,-3.151,-1.244,-0.005,0.135,0.248,0.921
cfo_to_assets,-3.803,-1.833,-0.793,0.029,0.187,0.289,0.533
revenue_to_assets,0.000,0.000,0.012,0.468,1.905,3.384,8.017
current_ratio,0.003,0.056,0.351,1.869,12.338,27.939,81.451


,financial_flag,cluster,major_sector,row_count,cluster_total,pct_of_cluster
2,Non-financial,0,Manufacturing / Industrials,4578,6593,0.694373
4,Non-financial,0,Services,1324,6593,0.200819
3,Non-financial,0,Mining / Energy,243,6593,0.036857
5,Non-financial,0,Transportation / Utilities,199,6593,0.030184
6,Non-financial,0,Wholesale / Retail,186,6593,0.028212
0,Non-financial,0,Agriculture,47,6593,0.007129
1,Non-financial,0,Construction,16,6593,0.002427
12,Non-financial,1,Transportation / Utilities,1315,4341,0.302926
9,Non-financial,1,Manufacturing / Industrials,1211,4341,0.278968
11,Non-financial,1,Services,753,4341,0.173462


## 6. Governance check: compare alternative k values

The selected five-cluster model is judged against nearby k alternatives for sanity, not automatic selection.

In [18]:
k_tests = evaluate_segments_k_range(
    model_df,
    segment_col=DEFAULT_SEGMENT_COL,
    k_values=range(2, 9),
    min_rows=DEFAULT_MIN_ROWS_PER_SEGMENT,
    cluster_only_segments=DEFAULT_TARGET_SEGMENTS,
)

k_tests

,segment,k,rows,features,feature_list,min_non_null_features,silhouette,calinski_harabasz,davies_bouldin
0,Non-financial,2,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.402994,17222.195827,1.094389
1,Non-financial,3,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.444893,16922.999951,0.917874
2,Non-financial,4,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.390756,15061.244816,1.156166
3,Non-financial,5,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.360757,13416.412028,1.211214
4,Non-financial,6,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.367783,12567.229226,1.255526
5,Non-financial,7,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.379692,12274.849922,1.169103
6,Non-financial,8,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.384871,11830.291236,1.113906


## 7. Save outputs and model artifact

Artifact schema/persistence lives in `artifacts.py`.

In [23]:
panel.to_parquet(CURRENT_OUTPUT_DIR / "issuer_year_feature_panel_v3.parquet", index=False)
clustered_panel.to_parquet(CURRENT_OUTPUT_DIR / f"clustered_panel_v3_by_{DEFAULT_SEGMENT_COL}.parquet", index=False)
cluster_profile_ranked.to_csv(CURRENT_OUTPUT_DIR / f"cluster_profile_v3_by_{DEFAULT_SEGMENT_COL}.csv", index=False)
cluster_profile_ranked_with_reps.to_csv(CURRENT_OUTPUT_DIR / f"cluster_profile_with_reps_v3_by_{DEFAULT_SEGMENT_COL}.csv", index=False)
cluster_medians.to_csv(CURRENT_OUTPUT_DIR / f"cluster_medians_v3_by_{DEFAULT_SEGMENT_COL}.csv")
feature_extremes.to_csv(CURRENT_OUTPUT_DIR / "feature_extremes_v3.csv")
industry_cluster_mix.to_csv(CURRENT_OUTPUT_DIR / f"industry_cluster_mix_v3_by_{DEFAULT_SEGMENT_COL}.csv", index=False)
metrics_df.to_csv(CURRENT_OUTPUT_DIR / f"cluster_metrics_v3_by_{DEFAULT_SEGMENT_COL}.csv", index=False)
k_tests.to_csv(CURRENT_OUTPUT_DIR / f"cluster_k_tests_v3_by_{DEFAULT_SEGMENT_COL}.csv", index=False)

print("Saved review outputs to:", CURRENT_OUTPUT_DIR)

Saved review outputs to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\credit_clustering_outputs_v3


In [26]:
artifact = build_credit_clustering_artifact(
    segment_artifacts=segment_artifacts,
    cluster_profile_ranked=cluster_profile_ranked,
    primary_segment="Non-financial",
    segment_col=DEFAULT_SEGMENT_COL,
    metrics_df=metrics_df,
    cluster_profile=cluster_profile,
    cluster_medians=cluster_medians,
    feature_extremes=feature_extremes,
    industry_cluster_mix=industry_cluster_mix,
    winsor_caps=None,
    artifact_version="v3_scorecard",
    notes=(
        "V3 KMeans++ k=5 model trained on bounded directional credit-risk factors. "
        "Debt-service risk includes EBITDA-enhanced diagnostics where available, "
        "with legacy coverage/FCF fallback when EBITDA is unavailable. "
        "Absolute size is excluded from clustering and retained only as diagnostic flags. "
        "Labels are rating-style interpretations, not formal credit ratings."
    ),
    extra_metadata={
        "model_name": "nonfinancial_credit_scorecard_kmeans_k5",
        "training_rows": int(len(clustered_panel[clustered_panel[DEFAULT_SEGMENT_COL] == "Non-financial"])),
        "source_notebook": "02_credit_clustering.ipynb",
    },
)

saved_path = save_artifact(
    artifact,
    ARTIFACT_PATH,
    segment="Non-financial",
)

print("Saved model artifact to:", saved_path)
summarize_artifact(artifact, segment="Non-financial")

Saved model artifact to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib


{'artifact_version': 'v3_scorecard',
 'primary_segment': 'Non-financial',
 'segment': 'Non-financial',
 'feature_cols': ['structural_distress_risk',
  'earnings_risk',
  'operating_cashflow_risk',
  'liquidity_risk',
  'leverage_risk',
  'debt_service_risk'],
 'n_clusters': 5,
 'cluster_labels': {2: '1 - Low risk / investment-grade-like',
  1: '2 - Moderate risk / lower-investment-grade-like',
  0: '3 - Elevated risk / leveraged',
  4: '4 - High risk / speculative',
  3: '5 - Distressed / near-default proxy'},
 'risk_rank': {2: 1, 1: 2, 0: 3, 4: 4, 3: 5},
 'cluster_sizes': {0: 6593, 1: 4341, 2: 7087, 3: 3350, 4: 2632},
 'silhouette': 0.3607570532446191,
 'calinski_harabasz': 13416.412028330398,
 'davies_bouldin': 1.2112136911495806}

## 8. Handoff to Notebook 03

Notebook 03 should load the saved artifact and use `score_companies()` plus `diagnostics.py` utilities for manual/private-company scoring.